# 27. 경제 시계열 예측: MLP와 LSTM 비교하기

이번 장에서는 다음 달 CPI 예측 문제를 두 방식으로 풀어봅니다.

```text
MLP  = lag feature 테이블 사용
LSTM = 과거 window 시계열 입력 사용
```

두 모델을 MAE/RMSE로 비교합니다.

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from keras.models import Sequential
from keras.layers import Input, Dense, Dropout, LSTM
from keras.callbacks import EarlyStopping

## 2. 평가 함수

In [ ]:
def regression_metrics(y_true, y_pred):
    """회귀 문제의 MAE와 RMSE를 계산합니다."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

## 3. 26장에서 만든 데이터셋 읽기

In [ ]:
DATA_PATH = Path(r"C:\work\deepLearning\deepLearning_textbook\data_outputs\fred_cpi_forecasting_dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "경제 예측 데이터셋 CSV가 없습니다. 먼저 26장 노트북을 실행해 CSV를 저장하세요."
    )

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values("date")

print("데이터 shape:", df.shape)
df.head()

## 4. MLP용 데이터 준비

In [ ]:
target_column = "target_next_cpi"

feature_columns = [col for col in df.columns if col not in ["date", target_column]]

X = df[feature_columns]
y = df[target_column]

split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_val = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_val = y.iloc[split_index:]

scaler_mlp = StandardScaler()
X_train_scaled = scaler_mlp.fit_transform(X_train)
X_val_scaled = scaler_mlp.transform(X_val)

print("MLP X_train shape:", X_train_scaled.shape)
print("MLP X_val shape:", X_val_scaled.shape)

## 5. MLP 모델 만들기

In [ ]:
input_dim = X_train_scaled.shape[1]

mlp_model = Sequential([
    Input(shape=(input_dim,)),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(16, activation="relu"),
    Dense(1)
])

mlp_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

mlp_model.summary()

## 6. MLP 학습

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

mlp_history = mlp_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stopping],
    verbose=1
)

## 7. MLP 평가

In [ ]:
mlp_pred = mlp_model.predict(X_val_scaled, verbose=0).ravel()

mlp_mae, mlp_rmse = regression_metrics(y_val, mlp_pred)

print(f"MLP MAE: {mlp_mae:.4f}")
print(f"MLP RMSE: {mlp_rmse:.4f}")

## 8. LSTM용 원본 지표 준비

In [ ]:
# LSTM에는 lag feature 테이블이 아니라 원래 경제 지표 흐름을 window로 만들어 넣습니다.
sequence_features = ["cpi", "unemployment_rate", "fed_funds_rate"]

sequence_values = df[sequence_features].values
target_values = df[target_column].values.reshape(-1, 1)

print("sequence_values shape:", sequence_values.shape)
print("target_values shape:", target_values.shape)

## 9. LSTM용 train/validation split과 스케일링

In [ ]:
train_seq = sequence_values[:split_index]
val_seq = sequence_values[split_index:]

train_target = target_values[:split_index]
val_target = target_values[split_index:]

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

train_seq_scaled = feature_scaler.fit_transform(train_seq)
val_seq_scaled = feature_scaler.transform(val_seq)

train_target_scaled = target_scaler.fit_transform(train_target)
val_target_scaled = target_scaler.transform(val_target)

print("train_seq_scaled shape:", train_seq_scaled.shape)
print("val_seq_scaled shape:", val_seq_scaled.shape)

## 10. LSTM window 데이터셋 만들기

In [ ]:
def make_sequence_dataset(features, target, window_size):
    """과거 window_size개월의 features로 다음 target을 맞히는 데이터셋을 만듭니다."""
    
    X = []
    y = []
    
    for i in range(len(features) - window_size):
        X.append(features[i:i + window_size, :])
        y.append(target[i + window_size])
    
    return np.array(X), np.array(y)


WINDOW_SIZE = 6

X_train_lstm, y_train_lstm = make_sequence_dataset(train_seq_scaled, train_target_scaled, WINDOW_SIZE)
X_val_lstm, y_val_lstm = make_sequence_dataset(val_seq_scaled, val_target_scaled, WINDOW_SIZE)

print("X_train_lstm shape:", X_train_lstm.shape)
print("y_train_lstm shape:", y_train_lstm.shape)
print("X_val_lstm shape:", X_val_lstm.shape)
print("y_val_lstm shape:", y_val_lstm.shape)

## 11. LSTM 모델 만들기

In [ ]:
lstm_model = Sequential([
    Input(shape=(WINDOW_SIZE, len(sequence_features))),
    LSTM(16),
    Dropout(0.2),
    Dense(1)
])

lstm_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

lstm_model.summary()

## 12. LSTM 학습

In [ ]:
lstm_history = lstm_model.fit(
    X_train_lstm,
    y_train_lstm,
    validation_data=(X_val_lstm, y_val_lstm),
    epochs=100,
    batch_size=8,
    callbacks=[early_stopping],
    verbose=1
)

## 13. LSTM 평가

In [ ]:
lstm_pred_scaled = lstm_model.predict(X_val_lstm, verbose=0)

# target_scaler는 CPI target을 0~1로 바꿨으므로 다시 원래 CPI 단위로 되돌립니다.
lstm_pred = target_scaler.inverse_transform(lstm_pred_scaled).ravel()
y_val_lstm_original = target_scaler.inverse_transform(y_val_lstm).ravel()

lstm_mae, lstm_rmse = regression_metrics(y_val_lstm_original, lstm_pred)

print(f"LSTM MAE: {lstm_mae:.4f}")
print(f"LSTM RMSE: {lstm_rmse:.4f}")

## 14. 결과표 비교

In [ ]:
comparison_df = pd.DataFrame([
    {"model": "MLP with lag features", "MAE": mlp_mae, "RMSE": mlp_rmse},
    {"model": "LSTM with 6-month window", "MAE": lstm_mae, "RMSE": lstm_rmse},
])

comparison_df.sort_values("MAE")

## 15. 실제값과 예측값 비교

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(y_val.values, label="actual CPI for MLP validation", linewidth=2)
plt.plot(mlp_pred, label="MLP prediction")

plt.title("MLP Actual vs Predicted CPI")
plt.xlabel("Validation sample")
plt.ylabel("CPI")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(y_val_lstm_original, label="actual CPI for LSTM validation", linewidth=2)
plt.plot(lstm_pred, label="LSTM prediction")

plt.title("LSTM Actual vs Predicted CPI")
plt.xlabel("Validation sample")
plt.ylabel("CPI")
plt.legend()
plt.tight_layout()
plt.show()

## 16. 해석 문장 만들기

In [ ]:
print("해석 템플릿")
print("- MLP는 lag feature를 사용해 과거 경제 지표를 2차원 표 형태 입력으로 받았다.")
print("- LSTM은 과거 6개월의 CPI, 실업률, 금리 흐름을 3차원 입력으로 받았다.")
print("- 두 모델을 같은 검증 구간에서 MAE와 RMSE로 비교했다.")
print("- 경제 월별 데이터는 샘플 수가 많지 않으므로 LSTM이 항상 MLP보다 유리하다고 단정할 수 없다.")
print("- 최종 모델 선택은 성능, 해석 가능성, 복잡도를 함께 고려해야 한다.")

## 17. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. MLP는 lag feature로 시계열 정보를 간접적으로 사용할 수 있다.
2. LSTM은 window 입력으로 시간 순서를 직접 다룬다.
3. 경제 데이터는 샘플 수가 적을 수 있어 복잡한 모델이 항상 유리하지 않다.
4. 모델은 MAE/RMSE 같은 같은 기준으로 비교해야 한다.
5. 성능뿐 아니라 복잡도와 설명 가능성도 모델 선택 기준이다.
```

## 과제

1. MLP와 LSTM의 입력 shape 차이를 설명해보세요.
2. `WINDOW_SIZE = 6`이 경제 데이터에서 어떤 의미인지 적어보세요.
3. LSTM이 MLP보다 성능이 낮게 나올 수 있는 이유를 설명해보세요.
4. CPI 예측에 추가하고 싶은 경제 지표를 3개 적어보세요.